# Session Explorer

This notebook allows focused exploration of specific chat sessions and user behavior patterns. Use it to:
- List all sessions and pick one to inspect
- View the full conversation thread within a session
- Analyze per-session timing and source patterns
- Compare sessions side by side

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown, HTML

LOG_PATH = Path("../logs/chat_interactions.jsonl")

records = []
with open(LOG_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Loaded {len(df)} interactions across {df['session_id'].nunique()} sessions")

## Session Directory
All sessions with their start time, number of questions, and duration.

In [ ]:
session_summary = df.groupby("session_id").agg(
    start_time=("timestamp", "min"),
    end_time=("timestamp", "max"),
    num_questions=("question", "count"),
    intents=("intent", lambda x: list(x)),
    avg_total_time=("total_time_seconds", "mean"),
    first_question=("question", "first"),
).sort_values("start_time", ascending=False)

session_summary["duration_minutes"] = (
    (session_summary["end_time"] - session_summary["start_time"]).dt.total_seconds() / 60
).round(1)

display_cols = ["start_time", "num_questions", "duration_minutes", "avg_total_time", "first_question"]
session_summary[display_cols].style.format({
    "avg_total_time": "{:.1f}s",
    "duration_minutes": "{:.1f} min",
})

## Inspect a Specific Session

Set `SESSION_ID` below to a session ID from the directory above, or set to `None` to use the most recent session.

In [ ]:
# --- Set the session to inspect ---
SESSION_ID = None  # Set to a specific session_id string, or None for most recent

if SESSION_ID is None:
    SESSION_ID = session_summary.index[0]  # most recent

session_df = df[df["session_id"] == SESSION_ID].sort_values("timestamp").reset_index(drop=True)
print(f"Session: {SESSION_ID}")
print(f"Questions: {len(session_df)}")
print(f"Time span: {session_df['timestamp'].min()} to {session_df['timestamp'].max()}")

### Conversation Thread
View the full Q&A exchange in this session.

In [ ]:
for i, row in session_df.iterrows():
    ts = row["timestamp"].strftime("%H:%M:%S")
    print("=" * 80)
    print(f"[{ts}]  Q{i+1}  |  intent: {row['intent']}  |  "
          f"retrieval: {row['retrieval_time_seconds']:.1f}s  |  "
          f"answer: {row['answer_time_seconds']:.1f}s  |  "
          f"total: {row['total_time_seconds']:.1f}s")
    print("=" * 80)
    print(f"\nQ: {row['question']}\n")
    print(f"A: {row['answer']}\n")

    # Show unique sources
    seen = set()
    for src in row["sources"]:
        pid = src.get("parent_id")
        if pid not in seen:
            seen.add(pid)
            print(f"   [{src.get('collection')}] {src.get('title')}")
    print()

### Session Timing Breakdown

In [ ]:
timing_data = session_df[["question", "intent", "retrieval_time_seconds",
                           "answer_time_seconds", "total_time_seconds"]].copy()
timing_data["question"] = timing_data["question"].str[:60]
timing_data.style.format({
    "retrieval_time_seconds": "{:.2f}s",
    "answer_time_seconds": "{:.2f}s",
    "total_time_seconds": "{:.2f}s",
})

### Sources Used in This Session
Which documents were retrieved across the whole session?

In [ ]:
session_sources = []
for _, row in session_df.iterrows():
    for src in row["sources"]:
        session_sources.append({
            "question": row["question"][:50],
            "parent_id": src.get("parent_id"),
            "title": src.get("title"),
            "collection": src.get("collection"),
        })

ssdf = pd.DataFrame(session_sources)
print(f"Total citations in session: {len(ssdf)}")
print(f"Unique documents: {ssdf['parent_id'].nunique()}")
print()
print("Document frequency:")
ssdf.groupby(["parent_id", "title", "collection"]).size().sort_values(ascending=False)

## Compare Sessions
Side-by-side metrics for all sessions.

In [ ]:
comparison = df.groupby("session_id").agg(
    questions=("question", "count"),
    unique_intents=("intent", "nunique"),
    avg_retrieval=("retrieval_time_seconds", "mean"),
    avg_answer=("answer_time_seconds", "mean"),
    avg_total=("total_time_seconds", "mean"),
    max_total=("total_time_seconds", "max"),
    unique_sources=("sources", lambda x: len(set(
        s.get("parent_id") for sources_list in x for s in sources_list
    ))),
    start=("timestamp", "min"),
).sort_values("start", ascending=False)

comparison.drop(columns=["start"]).style.format({
    "avg_retrieval": "{:.2f}s",
    "avg_answer": "{:.2f}s",
    "avg_total": "{:.2f}s",
    "max_total": "{:.2f}s",
})

## Search Interactions
Find specific questions or answers across all sessions.

In [ ]:
# --- Set search term ---
SEARCH_TERM = "ECT"  # Change this to search for different topics

mask = (
    df["question"].str.contains(SEARCH_TERM, case=False, na=False) |
    df["answer"].str.contains(SEARCH_TERM, case=False, na=False)
)
results = df[mask]
print(f"Found {len(results)} interactions matching '{SEARCH_TERM}'")
print()

for _, row in results.iterrows():
    print(f"[{row['session_id'][:20]}...]  {row['timestamp'].strftime('%Y-%m-%d %H:%M')}")
    print(f"  Q: {row['question']}")
    print(f"  Intent: {row['intent']}  |  Total: {row['total_time_seconds']:.1f}s")
    print()

## Excluded Parent IDs Analysis
Check if source exclusion is being used across sessions.

In [ ]:
df["has_exclusions"] = df["excluded_parent_ids"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
exclusion_count = df["has_exclusions"].sum()
print(f"Interactions with excluded sources: {exclusion_count} / {len(df)}")

if exclusion_count > 0:
    excluded = df[df["has_exclusions"]]
    print("\nExcluded parent IDs used:")
    for _, row in excluded.iterrows():
        print(f"  [{row['session_id'][:20]}...] Q: {row['question'][:50]}")
        print(f"    Excluded: {row['excluded_parent_ids']}")